In [ ]:
# =========================
# 02_baselines: FULL corrected, metrics-rich, Drive-aware baseline pipeline
# Paste this single cell into 02_baselines.ipynb and run it.
# =========================

import os, json, math
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from tqdm import tqdm
from packaging import version
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
    brier_score_loss, log_loss
)
import warnings
warnings.filterwarnings("ignore")

# ---------------- CONFIG (edit only if you must) ----------------
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2").resolve()
SPLITS_ROOT = DRIVE_PROJECT_ROOT / "data_splits"   # produced by 01_dataset_eda
OUTPUTS_ROOT = DRIVE_PROJECT_ROOT / "outputs" / "baselines"
TFIDF_PARAMS = {"ngram_range": (1,1), "max_features": 10000, "min_df": 2, "max_df": 0.98, "strip_accents": "unicode"}
# You can change TFIDF_PARAMS to match your original settings (e.g., max_features=50000) if you need exact reproduction.
LR_CONFIG = {"random_state": 42, "C": 1.0, "solver": "lbfgs", "max_iter": 2000}
SVM_CONFIG = {"random_state": 42, "C": 1.0, "max_iter": 10000, "dual": False}
FAST_SGD = False   # if True use SGDClassifier instead of LogisticRegression for speed
CALIBRATION_CV = 2  # cheaper calibration (set to 3 if you want stricter calibration at cost)
SAVE_PROBAS = True  # save per-class probability arrays as .npz when available
N_JOBS = 2
# ----------------------------------------------------------------

# safety: ensure output root exists
OUTPUTS_ROOT.mkdir(parents=True, exist_ok=True)

print("[INFO] sklearn version:", sklearn.__version__)
print("[INFO] SPLITS_ROOT:", SPLITS_ROOT)
print("[INFO] OUTPUTS_ROOT:", OUTPUTS_ROOT)
sklearn_ver = version.parse(sklearn.__version__)
use_estimator_kw = sklearn_ver >= version.parse("1.5.0")

def ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

def load_label_map_for_lang(lang):
    jm = SPLITS_ROOT / lang / "label_map.json"
    if not jm.exists():
        raise FileNotFoundError(f"Missing label_map.json for {lang} at {jm}. Run 01_dataset_eda.ipynb first.")
    jd = json.load(open(jm,encoding="utf8"))
    label_map = jd.get("label_map")
    inv_label_map = {int(v): str(k) for k, v in label_map.items()}
    return label_map, inv_label_map

def read_split_csv(lang, split_name):
    p = SPLITS_ROOT / lang / f"{split_name}.csv"
    if not p.exists():
        raise FileNotFoundError(f"Expected split file {p} not found. Run 01_dataset_eda.ipynb")
    return pd.read_csv(p)

def load_texts_from_df(df, drive_project_root=DRIVE_PROJECT_ROOT):
    if 'text' in df.columns:
        return df['text'].astype(str).tolist()
    if 'file_path' in df.columns:
        texts = []
        for fp in df['file_path'].tolist():
            if not isinstance(fp, str) or fp.strip()=="":
                texts.append(""); continue
            p = Path(fp)
            if p.exists():
                try:
                    texts.append(p.read_text(encoding="utf8", errors="ignore"))
                    continue
                except Exception:
                    pass
            alt = drive_project_root / fp
            if alt.exists():
                try:
                    texts.append(alt.read_text(encoding="utf8", errors="ignore"))
                    continue
                except Exception:
                    pass
            # try basename lookup under dataset folder
            candidate = None
            dataset_root = drive_project_root / "dataset"
            if dataset_root.exists():
                for sub in dataset_root.rglob(Path(fp).name):
                    candidate = sub
                    break
            if candidate and candidate.exists():
                try:
                    texts.append(candidate.read_text(encoding="utf8", errors="ignore"))
                    continue
                except Exception:
                    pass
            texts.append("")  # fallback
        return texts
    raise RuntimeError("CSV must include either 'text' or 'file_path' column.")

# ECE implementation (multiclass handled by taking max confidences for predicted class)
def expected_calibration_error(probs, labels, n_bins=15):
    """
    Compute ECE as in standard reliability diagram:
      - probs: numpy array shape (n_samples, n_classes) of predicted probabilities
      - labels: integer ground-truth labels shape (n_samples,)
    We'll compute ECE using predicted confidence = max predicted prob per sample.
    """
    if probs is None:
        return None
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lower, upper = bins[i], bins[i+1]
        mask = (confidences > lower) & (confidences <= upper)
        if np.sum(mask) == 0:
            continue
        acc = np.mean(predictions[mask] == labels[mask])
        avg_conf = np.mean(confidences[mask])
        ece += (np.sum(mask) / len(labels)) * abs(avg_conf - acc)
    return float(ece)

def mean_brier_score_multiclass(probs, labels):
    """
    Compute mean Brier score across classes (one-vs-all).
    For each class k, compute Brier score on binary (labels==k) vs predicted prob for class k.
    Then average across classes.
    """
    if probs is None:
        return None
    n_classes = probs.shape[1]
    scores = []
    for k in range(n_classes):
        y_true = (labels == k).astype(int)
        y_prob = probs[:, k]
        # brier_score_loss requires binary labels and probabilities
        try:
            bs = brier_score_loss(y_true, y_prob)
        except Exception:
            # numerical fallback
            bs = float(np.mean((y_prob - y_true)**2))
        scores.append(bs)
    return float(np.mean(scores))

# detect languages
if not SPLITS_ROOT.exists():
    raise FileNotFoundError(f"SPLITS_ROOT path does not exist: {SPLITS_ROOT}. Run 01_dataset_eda first.")
langs = sorted([d.name for d in SPLITS_ROOT.iterdir() if d.is_dir()])
if not langs:
    raise RuntimeError(f"No language directories found under {SPLITS_ROOT}. Run 01_dataset_eda first.")
print("[INFO] Languages found:", langs)

# main loop with progress bar
for lang in tqdm(langs, desc="Processing languages", unit="lang"):
    print("\n" + "="*80)
    print(f"[LANG] {lang}")
    lang_out = OUTPUTS_ROOT / lang
    baseline_dir = lang_out / "baseline"
    ensure_dir(baseline_dir)

    # load label maps
    label_map, inv_label_map = load_label_map_for_lang(lang)
    print(f"[INFO] label_map (str->id): {label_map}")

    # read splits
    train_df = read_split_csv(lang, "train")
    val_df   = read_split_csv(lang, "val")
    test_df  = read_split_csv(lang, "test")

    # load texts
    X_train_texts = load_texts_from_df(train_df)
    X_val_texts   = load_texts_from_df(val_df)
    X_test_texts  = load_texts_from_df(test_df)

    # map labels to ids
    def map_labels(df):
        if 'label_id' in df.columns:
            return [int(x) for x in df['label_id'].tolist()]
        if 'label' in df.columns:
            mapped = []
            for l in df['label'].tolist():
                key = str(l)
                if key in label_map:
                    mapped.append(int(label_map[key]))
                else:
                    found = None
                    for k in label_map.keys():
                        if k.lower() == key.lower():
                            found = label_map[k]; break
                    if found is not None:
                        mapped.append(int(found))
                    else:
                        mapped.append(-1)
            return mapped
        raise RuntimeError("CSV must have 'label' or 'label_id' column.")

    y_train = map_labels(train_df)
    y_val   = map_labels(val_df)
    y_test  = map_labels(test_df)

    # drop unknown label rows
    def filter_bad_rows(df, texts, labels, split_name):
        ok_idx = [i for i,l in enumerate(labels) if l != -1]
        if len(ok_idx) != len(labels):
            print(f"[WARN] {lang} {split_name}: {len(labels)-len(ok_idx)} rows with unknown label will be dropped.")
        return df.iloc[ok_idx].reset_index(drop=True), [texts[i] for i in ok_idx], [labels[i] for i in ok_idx]

    train_df, X_train_texts, y_train = filter_bad_rows(train_df, X_train_texts, y_train, "train")
    val_df,   X_val_texts,   y_val   = filter_bad_rows(val_df,   X_val_texts,   y_val,   "val")
    test_df,  X_test_texts,  y_test  = filter_bad_rows(test_df,  X_test_texts,  y_test,  "test")

    # fit tfidf
    print("[INFO] Fitting TF-IDF on train texts...")
    tfidf = TfidfVectorizer(**TFIDF_PARAMS)
    tfidf.fit(X_train_texts)
    X_train_tf = tfidf.transform(X_train_texts)
    X_val_tf   = tfidf.transform(X_val_texts)
    X_test_tf  = tfidf.transform(X_test_texts)

    # ------------------ Logistic Regression baseline ------------------
    print("[TRAIN] TF-IDF + LogisticRegression")
    if FAST_SGD:
        clf_lr = SGDClassifier(loss="log", max_iter=1000, tol=1e-3, random_state=42)
    else:
        clf_lr = LogisticRegression(**LR_CONFIG)
    clf_lr.fit(X_train_tf, y_train)

    yhat_lr = clf_lr.predict(X_test_tf)
    proba_lr = None
    try:
        proba_lr = clf_lr.predict_proba(X_test_tf)
    except Exception:
        proba_lr = None

    acc_lr = float(accuracy_score(y_test, yhat_lr))
    macro_f1_lr = float(f1_score(y_test, yhat_lr, average="macro", zero_division=0))
    rep_lr = classification_report(y_test, yhat_lr, zero_division=0, output_dict=True)
    cm_lr = confusion_matrix(y_test, yhat_lr)
    ece_lr = expected_calibration_error(proba_lr, np.array(y_test), n_bins=15) if proba_lr is not None else None
    brier_lr = mean_brier_score_multiclass(proba_lr, np.array(y_test)) if proba_lr is not None else None

    print(f"[RESULT LR] acc={acc_lr:.4f} macro_f1={macro_f1_lr:.4f} ECE={ece_lr} Brier={brier_lr}")

    # save LR artifacts
    lr_dir = baseline_dir / "logreg"
    ensure_dir(lr_dir)
    joblib.dump(clf_lr, lr_dir / "model.joblib")
    joblib.dump(tfidf, lr_dir / "tfidf.joblib")
    # save full probas if requested
    if proba_lr is not None and SAVE_PROBAS:
        np.savez_compressed(lr_dir / "test_probas.npz", probas=proba_lr)
    # predictions DF
    preds_lr = pd.DataFrame({
        "filename": test_df.get("filename", pd.Series([None]*len(y_test))),
        "file_path": test_df.get("file_path", pd.Series([None]*len(y_test))),
        "gold_label_id": y_test,
        "pred_label_id": list(map(int, yhat_lr))
    })
    preds_lr['gold_label_str'] = preds_lr['gold_label_id'].map(inv_label_map)
    preds_lr['pred_label_str'] = preds_lr['pred_label_id'].map(inv_label_map)
    preds_lr['pred_confidence'] = (np.max(proba_lr, axis=1).tolist() if proba_lr is not None else None)
    preds_lr.to_csv(lr_dir / "test_predictions.csv", index=False, encoding="utf8")

    metrics_lr = {
        "accuracy": acc_lr,
        "macro_f1": macro_f1_lr,
        "ece": ece_lr,
        "brier": brier_lr,
        "classification_report": rep_lr,
        "confusion_matrix": cm_lr.tolist()
    }
    with open(lr_dir / "metrics.json", "w", encoding="utf8") as f:
        json.dump(metrics_lr, f, indent=2, ensure_ascii=False)

    # ------------------ LinearSVC + calibration ------------------
    print("[TRAIN] TF-IDF + LinearSVC (calibrated)")
    svc = LinearSVC(**SVM_CONFIG)
    svc.fit(X_train_tf, y_train)

    # calibrate with robust argument name
    calib_kwargs = {"method": "sigmoid", "cv": CALIBRATION_CV}
    if use_estimator_kw:
        calibrated = CalibratedClassifierCV(estimator=svc, **calib_kwargs)
    else:
        calibrated = CalibratedClassifierCV(base_estimator=svc, **calib_kwargs)
    calibrated.fit(X_train_tf, y_train)

    yhat_svc = calibrated.predict(X_test_tf)
    proba_svc = None
    try:
        proba_svc = calibrated.predict_proba(X_test_tf)
    except Exception:
        proba_svc = None

    acc_svc = float(accuracy_score(y_test, yhat_svc))
    macro_f1_svc = float(f1_score(y_test, yhat_svc, average="macro", zero_division=0))
    rep_svc = classification_report(y_test, yhat_svc, zero_division=0, output_dict=True)
    cm_svc = confusion_matrix(y_test, yhat_svc)
    ece_svc = expected_calibration_error(proba_svc, np.array(y_test), n_bins=15) if proba_svc is not None else None
    brier_svc = mean_brier_score_multiclass(proba_svc, np.array(y_test)) if proba_svc is not None else None

    print(f"[RESULT SVC] acc={acc_svc:.4f} macro_f1={macro_f1_svc:.4f} ECE={ece_svc} Brier={brier_svc}")

    svc_dir = baseline_dir / "svc_calibrated"
    ensure_dir(svc_dir)
    joblib.dump(calibrated, svc_dir / "model.joblib")
    joblib.dump(tfidf, svc_dir / "tfidf.joblib")
    if proba_svc is not None and SAVE_PROBAS:
        np.savez_compressed(svc_dir / "test_probas.npz", probas=proba_svc)

    preds_svc = pd.DataFrame({
        "filename": test_df.get("filename", pd.Series([None]*len(y_test))),
        "file_path": test_df.get("file_path", pd.Series([None]*len(y_test))),
        "gold_label_id": y_test,
        "pred_label_id": list(map(int, yhat_svc))
    })
    preds_svc['gold_label_str'] = preds_svc['gold_label_id'].map(inv_label_map)
    preds_svc['pred_label_str'] = preds_svc['pred_label_id'].map(inv_label_map)
    preds_svc['pred_confidence'] = (np.max(proba_svc, axis=1).tolist() if proba_svc is not None else None)
    preds_svc.to_csv(svc_dir / "test_predictions.csv", index=False, encoding="utf8")

    metrics_svc = {
        "accuracy": acc_svc,
        "macro_f1": macro_f1_svc,
        "ece": ece_svc,
        "brier": brier_svc,
        "classification_report": rep_svc,
        "confusion_matrix": cm_svc.tolist()
    }
    with open(svc_dir / "metrics.json", "w", encoding="utf8") as f:
        json.dump(metrics_svc, f, indent=2, ensure_ascii=False)

    # summary file
    summary = {
        "language": lang,
        "n_train": len(y_train),
        "n_val": len(y_val),
        "n_test": len(y_test),
        "label_map": label_map,
        "inv_label_map": inv_label_map,
        "logreg_metrics": {"accuracy": acc_lr, "macro_f1": macro_f1_lr, "ece": ece_lr, "brier": brier_lr},
        "svc_metrics": {"accuracy": acc_svc, "macro_f1": macro_f1_svc, "ece": ece_svc, "brier": brier_svc}
    }
    with open(baseline_dir / "summary.json", "w", encoding="utf8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    print(f"[DONE] Baseline run for {lang} complete. Artifacts at {baseline_dir}")
    print("="*80)

print("\n[ALL DONE] Baselines completed for all languages.")


[INFO] sklearn version: 1.6.1
[INFO] SPLITS_ROOT: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/data_splits
[INFO] OUTPUTS_ROOT: /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/baselines
[INFO] Languages found: ['English', 'Hindi', 'Marathi']


Processing languages:   0%|          | 0/3 [00:00<?, ?lang/s]


[LANG] English
[INFO] label_map (str->id): {'G': 0, 'PG': 1, 'PG-13': 2, 'R': 3, 'NC-17': 4}
[INFO] Fitting TF-IDF on train texts...
[TRAIN] TF-IDF + LogisticRegression
[RESULT LR] acc=0.5581 macro_f1=0.2582 ECE=0.07475772878611534 Brier=0.11764455121793149
[TRAIN] TF-IDF + LinearSVC (calibrated)
[RESULT SVC] acc=0.5756 macro_f1=0.4757 ECE=0.048474435498098774 Brier=0.11374044072234668


Processing languages:  33%|███▎      | 1/3 [09:50<19:41, 590.63s/lang]

[DONE] Baseline run for English complete. Artifacts at /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/baselines/English/baseline

[LANG] Hindi
[INFO] label_map (str->id): {'U': 0, 'UA': 1, 'A': 2}
[INFO] Fitting TF-IDF on train texts...
[TRAIN] TF-IDF + LogisticRegression
[RESULT LR] acc=0.8710 macro_f1=0.8406 ECE=0.34509839345344046 Brier=0.13342476322053864
[TRAIN] TF-IDF + LinearSVC (calibrated)
[RESULT SVC] acc=0.8710 macro_f1=0.8406 ECE=0.13522054743539436 Brier=0.07475095469486857


Processing languages:  67%|██████▋   | 2/3 [11:24<04:58, 298.42s/lang]

[DONE] Baseline run for Hindi complete. Artifacts at /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/baselines/Hindi/baseline

[LANG] Marathi
[INFO] label_map (str->id): {'U': 0, 'UA': 1}
[INFO] Fitting TF-IDF on train texts...
[TRAIN] TF-IDF + LogisticRegression
[RESULT LR] acc=0.4375 macro_f1=0.4170 ECE=0.17980033119096528 Brier=0.24315724990440934
[TRAIN] TF-IDF + LinearSVC (calibrated)
[RESULT SVC] acc=0.5000 macro_f1=0.4667 ECE=0.10847599568667408 Brier=0.23353585188369058


Processing languages: 100%|██████████| 3/3 [12:19<00:00, 246.52s/lang]

[DONE] Baseline run for Marathi complete. Artifacts at /content/drive/MyDrive/PhDWorks/4_Final_Writing_Papers/04_JournalPaper_2/outputs/baselines/Marathi/baseline

[ALL DONE] Baselines completed for all languages.


### Analysis of Baseline Classification Code for Research Paper

Here is a technical analysis of the provided code cell, focusing on aspects relevant for inclusion in a research paper:

**What**

This code implements a baseline text classification pipeline for multiple languages. Its objective is to establish a performance reference for the task of classifying text documents into predefined categories based on their content. This step serves as a foundational comparison point for more complex or novel methods that may be developed later in the research.

**How**

The implementation utilizes a standard machine learning approach:

1.  **Text Representation:** Texts are transformed into numerical feature vectors using the **TF-IDF (Term Frequency-Inverse Document Frequency) vectorization** method. This process assigns weights to terms based on their frequency within a document and across the entire corpus, highlighting terms that are more distinctive to specific documents. The `TfidfVectorizer` is configured with `ngram_range`, `max_features`, `min_df`, and `max_df` to control the vocabulary size and filter out overly common or rare terms.
2.  **Classification Models:** Two widely used linear models are employed for classification:
    *   **Logistic Regression:** A linear model that uses a logistic function to model the probability of a binary outcome. For multi-class problems (as in this code), it typically uses a one-vs-rest (OvR) or multinomial approach.
    *   **Linear Support Vector Classifier (LinearSVC):** A linear model that finds a hyperplane that best separates different classes.
3.  **Probability Calibration:** The `LinearSVC` model, which doesn't inherently provide well-calibrated probabilities, is followed by **Platt Scaling (sigmoid calibration)** using `CalibratedClassifierCV`. This technique trains a logistic regression model on the output of the SVC to produce more reliable probability estimates.
4.  **Evaluation Metrics:** The performance of the models is evaluated using standard classification metrics:
    *   **Accuracy:** The proportion of correctly classified instances.
    *   **Macro F1-score:** The harmonic mean of precision and recall, calculated independently for each class and then averaged. This metric is useful for imbalanced datasets as it treats all classes equally.
    *   **Expected Calibration Error (ECE):** Measures the difference between the predicted confidence and the actual accuracy across different confidence bins. A lower ECE indicates better-calibrated probabilities.
    *   **Mean Brier Score:** Measures the mean squared difference between the predicted probabilities and the actual outcomes (represented as 0 or 1). A lower Brier score indicates better probability predictions.

**Why**

This baseline step is essential for the research for several reasons:

*   **Establishing a Lower Bound:** The performance of these relatively simple models provides a lower bound against which the performance of any proposed novel methods must be compared. A new method must outperform these baselines to demonstrate its effectiveness.
*   **Validation of Data and Setup:** Running baselines helps validate that the data splits are correctly formatted and that the overall experimental setup (loading data, training, evaluation) is functional before investing time in more complex models.
*   **Interpretability:** Linear models like Logistic Regression and LinearSVC are often more interpretable than complex deep learning models, providing insights into which features (TF-IDF terms) are most important for classification.

**Mathematical Formulation**

*   **TF-IDF:** The TF-IDF score for a term $t$ in a document $d$ in a corpus $D$ is typically calculated as:
    $$ \text{tfidf}(t, d, D) = \text{tf}(t, d) \times \text{idf}(t, D) $$
    where:
    *   $ \text{tf}(t, d) $ is the term frequency of term $t$ in document $d$ (e.g., raw count, log normalization).
    *   $ \text{idf}(t, D) = \log \frac{|D|}{| \{d' \in D : t \in d'\} |} $ is the inverse document frequency of term $t$ in the corpus $D$. $|D|$ is the total number of documents, and $| \{d' \in D : t \in d'\} |$ is the number of documents containing term $t$.

*   **Logistic Regression (for binary case):** The probability of a positive outcome ($y=1$) given feature vector $\mathbf{x}$ is modeled as:
    $$ P(y=1 | \mathbf{x}) = \sigma(\mathbf{w}^T \mathbf{x} + b) = \frac{1}{1 + e^{-(\mathbf{w}^T \mathbf{x} + b)}} $$
    where $\sigma$ is the sigmoid function, $\mathbf{w}$ is the weight vector, and $b$ is the bias term.

*   **Linear SVC Decision Function:** For a binary case, the decision is based on the sign of:
    $$ \mathbf{w}^T \mathbf{x} + b $$
    where $\mathbf{w}$ and $b$ are learned parameters.

*   **Expected Calibration Error (ECE):** Calculated by partitioning predictions into $M$ bins based on confidence and taking a weighted average of the difference between accuracy and average confidence in each bin:
    $$ \text{ECE} = \sum_{m=1}^{M} \frac{|B_m|}{n} |\text{acc}(B_m) - \text{conf}(B_m)| $$
    where $B_m$ is the set of indices of samples whose confidence falls into bin $m$, $n$ is the total number of samples, $\text{acc}(B_m)$ is the accuracy in bin $m$, and $\text{conf}(B_m)$ is the average confidence in bin $m$.

*   **Brier Score (for a single instance and class k):**
    $$ \text{BS}_k = (p_k - y_k)^2 $$
    where $p_k$ is the predicted probability for class $k$ and $y_k$ is 1 if the true label is $k$, and 0 otherwise. The mean Brier score is the average of this over all instances and classes.